# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 7: Large language model

### Nhóm 6:

- Hà Xuân Thiện - 24520031
- Hà Thanh Phong - 24520024
- Trần Quang Trường - 24521901

Ta sẽ sử dụng model distillBERT (được distill từ BERT cổ điển) trên dataset Imdb để đánh giá các review là positive hay negative

Demo sẽ được thực hiện chủ yếu qua thư viện transformer của huggingface

In [2]:
# ============================================================
# 1) Dependencies and Setup
# ============================================================
!pip -q install transformers datasets evaluate accelerate scikit-learn

import numpy as np
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)
import evaluate
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

set_seed(36)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "distilbert-base-uncased"

label_names = ["negative", "positive"]
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# ============================================================
# 2) Preprocessing
# - Loading the dataset
# - Setting up the tokenizer
# ============================================================
raw_dataset = load_dataset("imdb")

train_valid_split = raw_dataset["train"].train_test_split(
    test_size=0.1,
    seed=36
)

dataset = DatasetDict({
    "train": train_valid_split["train"],
    "validation": train_valid_split["test"],
    "test": raw_dataset["test"]
})

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 3) Metrics Setup
# ============================================================
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]

    return {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
    }

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})
Train size: 22500
Validation size: 2500
Test size: 25000

Sample example:
*SOILER* It's fake! The whole thing is a fake! There is no ghosts or zombies, Alan is a Lord and his cousin or brother or half brother or something like that wants the castle and his title for himself. So he invests this overly complicated and needless pointless plan ala SCOOBY-DOO to drive Alan to 
Label: 0


In [ ]:
# ============================================================
# 3) Model Training
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./bert_imdb_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\nTraining finished.")
print(train_result)

# ============================================================
# 4) Model Validation
# ============================================================
val_metrics = trainer.evaluate(tokenized_dataset["validation"])
print("\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.351302,0.266440,0.902400,0.872300,0.943300,0.906400
2,0.169743,0.342465,0.910000,0.900900,0.921800,0.911200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training finished.
TrainOutput(global_step=5626, training_loss=0.25913892705966857, metrics={'train_runtime': 458.184, 'train_samples_per_second': 98.214, 'train_steps_per_second': 12.279, 'total_flos': 2978382502759776.0, 'train_loss': 0.25913892705966857, 'epoch': 2.0})



Validation metrics:
eval_loss: 0.342464804649353
eval_accuracy: 0.91
eval_precision: 0.9009
eval_recall: 0.9218
eval_f1: 0.9112
eval_runtime: 5.8179
eval_samples_per_second: 429.705
eval_steps_per_second: 53.799
epoch: 2.0


In [1]:
# ============================================================
# 4) Model Evaluation
# ============================================================
test_output = trainer.predict(tokenized_dataset["test"])

y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=-1)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Score": [
        (y_pred == y_true).mean(),
        classification_report(y_true, y_pred, output_dict=True)["1"]["precision"],
        classification_report(y_true, y_pred, output_dict=True)["1"]["recall"],
        classification_report(y_true, y_pred, output_dict=True)["1"]["f1-score"],
    ]
})

metrics_df["Score"] = metrics_df["Score"].round(4)

print("Test set metrics")
print(metrics_df)

report = classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose().round(4)

print("\nDetailed classification report")
print(report_df)

NameError: name 'trainer' is not defined